# PolicyEngine tutorial — IMA World Congress 2026
## Analysing tax-benefit reform impacts with PolicyEngine

**Max Ghenis · PolicyEngine · max@policyengine.org**

This notebook is the *optional* Python companion to the tutorial. **You do not need it to follow along** — 
everything in the session can also be done in the browser at [policyengine.org](https://policyengine.org). 
This notebook is for participants who'd like to drive PolicyEngine from Python.

It runs entirely in **Google Colab** — nothing to install on your machine. Click 
**Runtime → Run all**, or run each cell with **Shift+Enter**.

We use the **US** model throughout, because its microdata is fully open. (The UK model works identically, 
but its population microdata is not publicly downloadable, so we demo UK population analysis in the web app.)

## 0. Setup
Install the unified `policyengine` package. This takes **~2–3 minutes** on Colab the first time — start it now. 
It installs both the US and UK tax-benefit models.

In [ ]:
%pip install -q policyengine
import policyengine as pe
print('PolicyEngine ready ✓')

## 1. A household calculation
`pe.us.calculate_household(...)` computes taxes, benefits and net income for any household — instantly, with no microdata.

Here: a married couple in Texas with two young children earning \$50,000.

In [ ]:
household = dict(
    people=[
        {'age': 40, 'employment_income': 50_000},
        {'age': 38, 'employment_income': 0},
        {'age': 8},
        {'age': 5},
    ],
    tax_unit={'filing_status': 'JOINT'},
    household={'state_code': 'TX'},
    year=2025,
)

base = pe.us.calculate_household(**household)
print(f"Net income:        ${base.household.household_net_income:,.0f}")
print(f"Federal income tax:${base.tax_unit.income_tax:,.0f}")
print(f"Child Tax Credit:  ${base.tax_unit.ctc:,.0f}")
print(f"SNAP:              ${base.spm_unit.snap:,.0f}")

## 2. Model a reform
Reforms are just a dict of `{parameter: new_value}`. Let's raise the maximum Child Tax Credit from its 2025 value of \$2,200 to \$3,000 per child, and see the effect on this family.

In [ ]:
reform = {'gov.irs.credits.ctc.amount.base[0].amount': 3_000}

reformed = pe.us.calculate_household(**household, reform=reform)

gain = reformed.household.household_net_income - base.household.household_net_income
print(f"Baseline CTC:      ${base.tax_unit.ctc:,.0f}")
print(f"Reform CTC:        ${reformed.tax_unit.ctc:,.0f}")
print(f"Net income change: ${gain:,.0f}")

## 3. How the reform varies with earnings
Because the calculator is instant, we can sweep it across earnings to see how the reform's value changes — 
a quick way to surface phase-ins and phase-outs without any microdata.

In [ ]:
import numpy as np, plotly.graph_objects as go

earnings = list(range(0, 200_001, 10_000))
gains = []
for e in earnings:
    hh = dict(household); hh['people'] = [
        {'age': 40, 'employment_income': e}, {'age': 38},
        {'age': 8}, {'age': 5}]
    b = pe.us.calculate_household(**hh)
    r = pe.us.calculate_household(**hh, reform=reform)
    gains.append(r.household.household_net_income - b.household.household_net_income)

fig = go.Figure(go.Scatter(x=earnings, y=gains, mode='lines', line=dict(color='#2C6496', width=3)))
fig.update_layout(title='Household gain from a $3,000 CTC (married, 2 children, TX)',
                  xaxis_title='Employment income ($)', yaxis_title='Net income gain ($)',
                  template='plotly_white', height=420)
fig.show()

## 4. Population analysis — runs in Colab

A national microsimulation over the full ~573,000-person dataset is memory-heavy. But PolicyEngine ships 
**per-state** datasets that are light enough to run live here — a full baseline-vs-reform comparison for one 
state takes about half a minute and a few GB of RAM.

> For the full national picture (budget, distribution, poverty, winners and losers across all 50 states), the 
web app at [policyengine.org](https://policyengine.org) runs it server-side in seconds.

In [ ]:
from policyengine_core.reforms import Reform

STATE = 'states/DC'   # try states/VT, states/WY, states/AK, ... (small states keep memory low)
YEAR = 2025

# Baseline: the managed simulation uses PolicyEngine's canonical Populace microdata
baseline = pe.us.managed_microsimulation(dataset=STATE)
base_net = baseline.calculate('household_net_income', YEAR).sum()
people = baseline.calculate('age', YEAR).count()
print(f'{STATE}: {people/1e6:.2f}M people; baseline net income ${base_net/1e9:,.1f}B')

# Reform: raise the maximum CTC to $3,000 (population API takes a Reform object)
reform = Reform.from_dict(
    {'gov.irs.credits.ctc.amount.base[0].amount': {'2025-01-01.2025-12-31': 3000}},
    country_id='us',
)
reformed = pe.us.managed_microsimulation(dataset=STATE, reform=reform)
cost = reformed.calculate('household_net_income', YEAR).sum() - base_net
print(f'State-level cost of a $3,000 CTC: ${cost/1e9:,.2f}B')

## 5. Where to go next
- **Web app — [policyengine.org](https://policyengine.org):** the fastest way to do full population reform analysis 
(budget, distribution, poverty, winners/losers) for the US and UK. No setup.
- **AI assistant:** PolicyEngine ships a Claude Code plugin that drives all of the above from plain-language prompts 
("What's the poverty effect of a $3,000 CTC?") — demoed live in the session.
- **Docs:** [github.com/PolicyEngine/policyengine.py](https://github.com/PolicyEngine/policyengine.py)
- **Questions:** max@policyengine.org